In [2]:
from faster_whisper import WhisperModel

In [3]:
# 모델
model = WhisperModel(
    "base",
    device="cuda",
    compute_type="float16"
)

# 텍스트 변환
segments, info = model.transcribe(
    "./audios/meeting.mp3",
    language="ko",
    beam_size=5 # 높을수록 정확하지만 느려진다(default = 5)
)

print(segments)
full_text = ""
for seg in segments:
    print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}")
    full_text += seg.text
print(f"인식된 텍스트 : {full_text}")
print(f"감지된 언어: {info.language} (확률: {info.language_probability: .0%})")

<generator object WhisperModel.generate_segments at 0x000001986789CA40>
[0.00s -> 3.60s]  다들 도착하셨나요? 그럼 회의 시작할게요.
[3.60s -> 6.52s]  먼저 디자인팀 상황부터 공유해주시겠어요?
[6.52s -> 11.96s]  네, 디자인팀은 이번 주까지 시안 1차 수정본 제출목별로 작업 중입니다.
[11.96s -> 15.96s]  주요 피드백 반영했고, 마감은 목을까지 가능합니다.
[15.96s -> 18.40s]  좋아요. 개발팀은요?
[18.40s -> 22.04s]  기능 개발은 80% 열됐습니다.
[22.04s -> 25.40s]  요그인 기능과 계십한 기능은 이번 주에 마무리할 예정이고
[25.40s -> 28.56s]  다음 주월을부터 내부 테스트를 시작하려고 합니다.
[28.56s -> 30.84s]  일정대로 잘 진행되고 있네요.
[30.84s -> 32.64s]  마케팅 쪽은 어떤가요?
[32.64s -> 36.20s]  마케팅팀은 월초에 있을 프로모션을 준비 중입니다.
[36.20s -> 40.40s]  다만 이번 주 안으로 기획 초원을 작성하는 건 조금 타이트할 것 같은데
[40.40s -> 45.12s]  가능하다면 기획 초원 제출 기안을 다음 주 수요일까지로 도정해 주실 수 있을까요?
[45.12s -> 49.32s]  음, 알겠어요. 전체 일정에는 영향이 크지 않으니
[49.32s -> 51.84s]  다음 주 수요일까지로 조정합시다.
[51.84s -> 54.88s]  그럼 지금 남은 과제는 기능 테스트 계획 수리
[54.88s -> 60.24s]  디자인 수정 최종본 검토 마케팅 기획 초원 전검 2세 가지로 정리할 수 있겠네요?
[60.24s -> 64.64s]  네, 테스트 계획 문서는 이번 주 금요일까지 작성해서 공유하겠습니다.
[64.64s -> 69.12s]  디자인 수정보는 목요일에 완성되는 대로 바로 공유 드리겠습니다.
[69.12s 

In [4]:
from openai import OpenAI

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model = "gpt-4o-mini-tts",
    voice="verse",
    input="""
미국 대통령이 이란의 에너지·전기 시설에 대한 공격을 닷새 뒤로 늦추면서 이란과 협상중이라고 밝혔지만, 중동 지역에는 여전히 폭탄이 떨어지고 있다. 이스라엘은 이란과 레바논에 대한 공격을 이어갔고, 이란 역시 호르무즈해협을 완전히 통제하고 있다고 과시하며 이스라엘과 페르시아만 국가들에 대한 공격을 멈추지 않았다.
'48시간 최후통첩'을 보낸 도널드 트럼프 미국 대통령이 공격을 닷새 동안 연기하면서 이란과 협상중이라고 밝힌 뒤인 23일 오후(이스라엘 시각) 베냐민 네타냐후 총리는 영상 성명을 통해 트럼프 대통령과 종전 협상에 대해 논의했다고 확인했다.
네타냐후 총리는 "트럼프 대통령은 합의를 통해 전쟁을 목표를 달성하기 위해 우리가 미군과 함께 이룩한 엄청난 성과들을 활용할 기회가 있다고 믿고 있다. 그같은 합의는 우리의 핵심 이익을 보호할 것"이라고 말했다. 이란과의 협상에서 미국이 이스라엘의 요구를 충실히 반영할 것이라는 얘기다.
하지만 그는 이란과 레바논 헤즈볼라에 대한 공격을 계속하고 있다고 밝혔다. 그는 "우리는 미사일 프로그램과 핵 프로그램을 무력화시키고 있으며, 헤즈볼라에 대한 강력한 공격을 계속하고 있다"고 말했다.
중동 지역에서 취재를 하고 있는 여러 외신의 보도를 종합하면, 트럼프 대통령의 '공격 닷새 연기' 발표 이후에도 이스라엘은 레바논 베이루트, 티레, 나바티에와 테헤란을 공습했다. 이란은 예루살렘과 북이스라엘 지역, 쿠웨이트 등을 미사일·드론으로 공격했다.
""",
    speed=1.2,  # 0.25~ 4.0
    instructions="슬프고 비통한 목소리로 말해줘."
) as response:
    response.stream_to_file("./audios/이나원_news.mp3")

In [5]:
# 오디오 백그라운드 실행
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3("./audios/이나원_news.mp3")
play(sound)